# Question 2: do converts bite as often as the vampires the walkers recruited?

MATH 1660, November 2026.

A **root** is a vampire nobody bit: a player recruited in person by a walker, or a player who joined from a poster. A **convert** is a vampire who was bitten by another vampire. Did converts bite anyone as often as roots did? This is the permutation test of November 4, covered in class.

**How this connects to *does branching theory describe our game below the threshold?*** Every branching model assumes that each generation bites from the same distribution: a convert bites like the vampire who bit them. This notebook tests that assumption. With few converts it can reject the assumption, but it cannot confirm it.

Your plan makes its own choices for this test, such as whether roots who joined from a poster are counted, and the notebook computes the test with them.

**Use practice mode to write your plan.** The data from our game are in the **research export**: a set of CSV files made by the game's platform, recording who played, who bit whom and when, and when each player turned and burned out. Each team is given a web link from which this notebook can read those files, but only after the team's plan is submitted.

Until then, the notebook does not read our game's data. While `PRACTICE = True` in Step 1, it makes up an export for an imaginary game, with the same files and the same columns, and runs every step on that. Use this to learn what each step computes: run the notebook, change the practice settings in Step 1, and run it again. Then decide the choices in Step 1 that your plan will state, such as `POSTER_ROOTS_IN`.

After your plan is submitted, you are given the link. Paste it into Step 1, set `PRACTICE = False`, and run the notebook again: it now reads our game's export, and uses the choices your plan states.

**The result from the choices in your plan is your test.** Suppose that after seeing it you run the notebook again with a different choice, for example a different `POSTER_ROOTS_IN` in Step 1. The result of that second run is **exploratory**, and your report says so. It gives the result from your plan's choices first. Then it gives the second result, labeled *exploratory*, with the choice you changed and why. An exploratory result is not a test. Its p-value is not the chance that the test describes, because the choice was made after seeing the data, and a choice made that way can be the one that happens to give the smallest p-value. An exploratory result can still suggest a question worth testing in another game.

Nobody has checked this notebook except its author. If a number looks wrong, it may be.

## How to use this notebook

1. **Make your own copy first:** *File*, then *Save a copy in Drive*.
2. Run the cells in order, from the top: click inside a cell and press **Shift+Enter**.
3. Change only the values in Step 1.

## Step 1: your choices, and the export

- `PRACTICE`: while it is `True`, the notebook makes up an export for an imaginary game and does not read our game's export. `PRACTICE_SEED` picks which made-up export you get; change it to get another.
- `PRACTICE_CONVERTS`: how the made-up converts bite.
  - `"same"`: every vampire's number of bites is drawn the same way, roots and converts alike, so the null hypothesis of Step 5 is true.
  - `"never"`: roots bite as in `"same"`, and converts bite no one.

  In both, about 1 root in 5 bites someone. Run both, and several seeds of each, and see how often the test finds the difference when there is one.
- `EXPORT_LINK`: empty until your plan is submitted. Then paste your team's export link between the quotes. It ends with `/`. **Do not share a notebook that has the link in it**: anyone with the link can read the research data. If you would rather upload the files, leave the link empty and upload `settings.csv`, `participants.csv`, `transmissions.csv`, and `state_changes.csv` with the folder icon on the left.
- `POSTER_ROOTS_IN`: `True` counts the roots who joined from a poster, with the roots a walker recruited; `False` leaves them out, and the chains they started with them. Your plan states which, and the other is a check.
- `REPS`: how many random lists Step 6 draws.

In [ ]:
# Step 1: your choices
PRACTICE = True            # True: a made-up export. False: our export (only after your plan is submitted)
PRACTICE_SEED = 1          # change it to get a different made-up export
PRACTICE_CONVERTS = "same" # "same" or "never": how the made-up converts bite
EXPORT_LINK = ""           # after your plan is submitted: paste your team's export link here
POSTER_ROOTS_IN = True     # True: roots from posters are counted. False: they and their chains are left out
REPS = 20000               # random lists drawn in Step 6

Run the next three cells. The first two copy one file of the course simulator, `vampsim/compare.py`, into this session, as of November 2026; you do not need to read it. The tests below call two functions from it: `compare.p_trees` (Step 5) and `compare.p_trees_sampled` (Step 6). The third reads the export, or makes one up.

In [ ]:
# Step 1b: make a folder for the simulator file
import os
os.makedirs("vampsim", exist_ok=True)
open("vampsim/__init__.py", "w").close()
print("Folder ready.")

In [ ]:
%%writefile vampsim/compare.py
"""Whether one group of vampires bites less often than another, and how many
vampires it takes to tell.

Added 2026-10-25 for the week-11 question *do converts bite as often as seeds?*
The unit is a vampire whose five days are over, and the outcome is whether they
bit at least one person, not how many: bites come in groups, so a vampire who
bit three friends at one sitting made one decision to show the code.

The test is the shuffle test of week 6: shuffle the group labels among all the
vampires and count how often group 2 gets as few biters as it did. Its exact
form is the one-sided Fisher test, computed here by the hypergeometric
distribution. Group 1 is the reference (the seeds), group 2 the group suspected
of biting less (the converts). Exact enumeration, no simulation.

Corrected 2026-10-30: for seeds against converts this shuffle is wrong. It
counts arrangements that cannot happen, such as no seed biting while converts
bite, although converts exist only because someone bit them, and so its p-value
is too small. It also treats the
number of converts as free, when it is the number of bites. p_value, power,
power_given and needed are kept for two groups whose sizes do not depend on
each other, such as seeds against poster joiners; for roots against converts
use p_trees or p_trees_sampled, below.
"""

from __future__ import annotations

import random
from math import comb
from typing import List, Optional, Sequence


def p_value(x1: int, n1: int, x2: int, n2: int) -> float:
    """One-sided p-value that group 2 has fewer biters: x1 of n1 against x2 of n2.
    The chance a shuffle of the labels leaves group 2 with x2 biters or fewer."""
    t, n = x1 + x2, n1 + n2
    return sum(comb(n2, j) * comb(n1, t - j) for j in range(x2 + 1)
               if 0 <= t - j <= n1) / comb(n, t)


def _binom(n: int, p: float) -> List[float]:
    return [comb(n, x) * p ** x * (1 - p) ** (n - x) for x in range(n + 1)]


def power(p1: float, n1: int, p2: float, n2: int, alpha: float = 0.05) -> float:
    """Chance the test finds the difference at level alpha, when each vampire in
    group i bites with chance pi, independently."""
    b1, b2 = _binom(n1, p1), _binom(n2, p2)
    return sum(b1[x1] * b2[x2] for x1 in range(n1 + 1) for x2 in range(n2 + 1)
               if p_value(x1, n1, x2, n2) < alpha)


def power_given(x1: int, n1: int, p2: float, n2: int, alpha: float = 0.05) -> float:
    """As power(), with group 1 already observed: x1 biters of n1. The seeds are
    finished before the converts are, so this is the version that applies."""
    return sum(b for x2, b in enumerate(_binom(n2, p2)) if p_value(x1, n1, x2, n2) < alpha)


def needed(x1: int, n1: int, x2: int = 0, alpha: float = 0.05,
           n_max: int = 1000) -> Optional[int]:
    """The smallest group 2 in which x2 biters is a finding at level alpha,
    given x1 biters of n1 in group 1; None if no size up to n_max is enough."""
    for n2 in range(max(x2, 1), n_max + 1):
        if p_value(x1, n1, x2, n2) < alpha:
            return n2
    return None


# Roots against converts (added 2026-10-30). A root is a vampire nobody bit (a
# seed or a poster joiner), a convert one bitten by another vampire, a biter one
# who bit at least once. Write r roots, V vampires in all, so V - r converts,
# which is also the number of bites.
#
# The null hypothesis: every vampire's bite count is drawn independently from
# one distribution, of any shape, roots and converts alike. List the roots
# first, then the converts in the order they were bitten. A list of counts is
# *possible* if every convert was bitten by someone above them in it:
# r + sum over the first m entries of (count - 1) > 0 for every m < V. Each
# possible list is one set of trees, with probability the product of its
# counts' probabilities, so given the counts every possible list is equally
# likely. The test asks how often a possible list puts as many biters among
# the first r entries (the roots) as observed.
#
# Assumed besides the null: every count is final (every vampire's five days are
# over), and the day does not matter (converts are always vampires later than
# the roots who bit them, so a game that bites less as it gets older looks the
# same as converts who bite less).

def possible(counts: Sequence[int], r: int) -> bool:
    """Whether a list of bite counts, roots first, could have produced trees."""
    q = r
    for x in counts[:-1]:
        q += x - 1
        if q <= 0:
            return False
    return True


def p_trees(r: int, b: int, V: int) -> float:
    """Chance that all b biters are roots, over the possible lists, when no
    convert bit: C(r-1, b-1) / C(V-1, b-1). By the cycle lemma, a fraction r/V
    of all lists are possible, and every list with all biters among the roots
    is one of them, so this is the plain shuffle's C(r, b) / C(V, b) times V/r.
    Conditional on the observed counts, as is p_trees_sampled: only lists made of
    those counts are compared, since under the null every such list has the same
    chance whatever the bite-count distribution (note added 2026-11-04)."""
    return comb(r - 1, b - 1) / comb(V - 1, b - 1)


def p_trees_sampled(root_counts: Sequence[int], convert_counts: Sequence[int],
                    reps: int = 20000, rng: Optional[random.Random] = None) -> float:
    """One-sided p-value that converts bite less than roots, for any counts:
    the share of possible lists, drawn at random, with at least as many biters
    among the roots as observed. A list is drawn by shuffling all the counts and
    taking one of its r rotations that are possible, at random (the cycle
    lemma: a list whose counts less 1 sum to -r has exactly r of them), which
    makes every possible list equally likely."""
    rng = rng or random.Random()
    r = len(root_counts)
    counts = list(root_counts) + list(convert_counts)
    V = len(counts)
    if sum(counts) != V - r:
        raise ValueError("the number of converts must equal the number of bites")
    observed = sum(x > 0 for x in root_counts)
    hits = 0
    for _ in range(reps):
        rng.shuffle(counts)
        # S[t] = sum of (count - 1) over the first t entries. The rotation that
        # starts at entry s is possible when S[s] is below every S[t], t < s,
        # and at most r - 1 above every S[t], t > s.
        S = [0]
        for x in counts:
            S.append(S[-1] + x - 1)
        before = [0] * V
        lowest = float("inf")
        for s in range(V):
            before[s] = lowest
            lowest = min(lowest, S[s])
        starts, lowest = [], float("inf")
        for s in range(V - 1, -1, -1):
            lowest = min(lowest, S[s + 1])
            if S[s] < before[s] and S[s] <= lowest + r - 1:
                starts.append(s)
        s = rng.choice(starts)
        hits += sum(x > 0 for x in (counts[s:] + counts[:s])[:r]) >= observed
    return hits / reps

In [ ]:
# Step 1c: read the export, or make one up
import csv, io, math, random, urllib.request
from datetime import datetime, timedelta, timezone

FILES = ["settings.csv", "participants.csv", "transmissions.csv", "state_changes.csv"]

def read(name):
    if EXPORT_LINK:
        text = urllib.request.urlopen(EXPORT_LINK + name).read().decode("utf-8")
    else:
        text = open(name, encoding="utf-8").read()
    return list(csv.DictReader(io.StringIO(text)))

t = lambda s: datetime.fromisoformat(s) if s else None
hours = lambda a, b: (b - a).total_seconds() / 3600     # hours from a to b
iso = lambda d: d.isoformat()

def show_p(p, reps=None):
    # a p-value with 4 decimals, a very small one in powers of 10, and a simulated 0 as "below 1/reps"
    if reps and p == 0:
        return f"0 (below 1/{reps})"
    return f"{p:.4f}" if p >= 0.001 else f"{p:.2e}"
LAUNCH = datetime(2026, 10, 16, 23, 0, tzinfo=timezone.utc)   # used only by the made-up export
SETTINGS = {"spree_start_at": "2026-11-09T18:03:00+00:00", "end_at": "2026-11-14T00:00:00+00:00",
            "refreshed_at": "2026-11-14T00:00:00+00:00"}
import importlib, vampsim.compare
importlib.reload(vampsim.compare)
from vampsim import compare

def made_up_export(seed):
    # 40 roots (34 recruited by a walker, 6 from a poster), who turn over the first 14 days.
    # Each vampire bites 0, 1, 2, or 3 people (0 for about 4 in 5), at random times in a 120-hour
    # lifetime; a convert turns 12 hours after being bitten. Some chains are still going
    # when the spree starts, as in the real game.
    rng = random.Random(seed)
    out = {"settings.csv": [dict(SETTINGS)], "participants.csv": [], "transmissions.csv": [],
           "state_changes.csv": []}
    queue = []
    for i in range(40):
        turn = LAUNCH + timedelta(hours=rng.uniform(0, 14 * 24))
        queue.append((f"practice-{i + 1:02d}", turn, turn, "walker" if i < 34 else "poster"))
    while queue:
        pid, joined, turn, how = queue.pop(0)
        out["participants.csv"].append({"participant_id": pid, "is_seed": str(how != ""),
                                        "joined_through": how, "joined_at": iso(joined),
                                        "turn_at": iso(turn)})
        out["state_changes.csv"].append({"participant_id": pid, "change": "vampire → burned out",
                                         "at": iso(turn + timedelta(hours=120))})
        if how == "" and PRACTICE_CONVERTS == "never":
            continue
        for j in range(rng.choice([0] * 19 + [1, 2, 2, 3, 3])):
            at = turn + timedelta(hours=rng.uniform(0, 120))
            target = f"{pid}.{j + 1}"
            out["transmissions.csv"].append({"source_id": pid, "target_id": target, "at": iso(at),
                                             "target_already_in_game": "False"})
            queue.append((target, at, at + timedelta(hours=12), ""))
    return out

data = made_up_export(PRACTICE_SEED) if PRACTICE else {name: read(name) for name in FILES}
settings = data["settings.csv"][0]
players = {p["participant_id"]: p for p in data["participants.csv"]}
rows = data["transmissions.csv"]
if PRACTICE:
    print(f"PRACTICE MODE: a made-up export (seed {PRACTICE_SEED}). None of these numbers comes from our game.")
else:
    print("Our export, made at", settings["refreshed_at"], "(UTC)")
print(len(players), "players,", len(rows), "rows in transmissions.csv")

## Step 2: finished vampires, and their bites

A vampire is **finished** when their five days are over, so their number of bites can no longer change. The spree, which started at `spree_start_at` in `settings.csv`, let burned-out players bite again, so a vampire counts as finished here only if they burned out *before the spree started*. Use `state_changes.csv` for this, not the `burnout_at` column of `participants.csv`: for players the spree brought back, `burnout_at` shows the end of the game, not the time they first burned out.

A **bite** is a row of `transmissions.csv` that made its target a new player (`target_already_in_game` is `False`). A row with `True` is a scan of someone already in the game, and is not a bite. A finished vampire's bites are the ones they made **before their first burnout**; a bite made during the spree, after the spree brought them back, is left out.

In [ ]:
# Step 2: finished vampires, and the bites each made before their first burnout
spree = t(settings["spree_start_at"])
first_burnout = {}
for row in data["state_changes.csv"]:
    if row["change"] == "vampire → burned out":
        first_burnout.setdefault(row["participant_id"], t(row["at"]))

finished = [p for p in players.values()
            if p["participant_id"] in first_burnout and first_burnout[p["participant_id"]] < spree]
finished_ids = {p["participant_id"] for p in finished}

bites_by = {}      # participant_id -> times of their bites before their first burnout
recruiter = {}     # participant_id of a convert -> participant_id of the vampire who bit them
for b in rows:
    if b["target_already_in_game"] != "False":
        continue
    recruiter[b["target_id"]] = b["source_id"]
    src = b["source_id"]
    if src in first_burnout and t(b["at"]) < first_burnout[src]:
        bites_by.setdefault(src, []).append(t(b["at"]))

def bites_of(pid):
    return len(bites_by.get(pid, []))

print(len(finished), "finished vampires;",
      sum(bites_of(p["participant_id"]) > 0 for p in finished), "of them bit someone;",
      sum(bites_of(p["participant_id"]) for p in finished), "bites between them")

## Step 3: the chains that are counted

Follow each convert back to the vampire who bit them, and that vampire back to whoever bit them, until a root is reached. A root and everyone reached back to them form one **chain**.

**A chain is counted only if every vampire in it is finished.** A vampire who is not finished can still bite, and a bite would add a convert to the chain, so the chain's counts are not yet final. If `POSTER_ROOTS_IN` is `False`, a chain whose root joined from a poster is also left out, with every convert in it.

For each vampire $i$ in a counted chain:

- $y_i$ is the number of people vampire $i$ bit (Step 2);
- $Z_i = 1$ if $y_i \ge 1$, and $Z_i = 0$ otherwise. A vampire with $Z_i = 1$ is a **biter**.

Only $Z_i$ is used, not $y_i$: a vampire who bit three friends at one sitting made one decision to show their code, so whether they bit at all is what the test compares. So how bites at one sitting are counted does not matter here.

Then, over the counted chains:

- $r$ is the number of roots and $m$ the number of converts; $V = r + m$ is the number of vampires.
- Each bite made one convert, so the total of the $y_i$ is $m$. The cell checks this.
- $T$ is the number of roots with $Z_i = 1$, the **root biters**, and $b$ is the number of biters in all, roots and converts.

In [ ]:
# Step 3: the counted chains, and r, m, T, b
def root_of(pid):
    while pid in recruiter:
        pid = recruiter[pid]
    return pid

chains_by_root = {}
for pid in players:
    chains_by_root.setdefault(root_of(pid), []).append(pid)

counted, left_out = {}, {"not finished": 0, "poster root": 0}
for root, members in chains_by_root.items():
    if not all(pid in finished_ids for pid in members):
        left_out["not finished"] += 1
    elif not POSTER_ROOTS_IN and players[root]["joined_through"] == "poster":
        left_out["poster root"] += 1
    else:
        counted[root] = members

root_counts = [bites_of(root) for root in counted]
convert_counts = [bites_of(pid) for root, members in counted.items() for pid in members if pid != root]
r, m = len(root_counts), len(convert_counts)
V = r + m
T = sum(x >= 1 for x in root_counts)
convert_biters = sum(x >= 1 for x in convert_counts)
b = T + convert_biters

print(f"chains counted: {len(counted)}; left out because a vampire in them is not finished: "
      f"{left_out['not finished']}; left out as poster roots: {left_out['poster root']}")
print(f"{'':>9}  {'vampires':>8}  {'biters':>6}  {'bites':>5}")
print(f"{'roots':>9}  {r:>8}  {T:>6}  {sum(root_counts):>5}")
print(f"{'converts':>9}  {m:>8}  {convert_biters:>6}  {sum(convert_counts):>5}")
print(f"r = {r}, m = {m}, V = {V}, T = {T} root biters, b = {b} biters in all")
print("check, total bites = number of converts:", sum(root_counts) + sum(convert_counts) == m)

## Step 4: the figure

Two bars: the share of roots who bit someone and the share of converts who bit someone, with the counts written on each bar.

In [ ]:
# Step 4: the figure
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 4))
shares = [T / r if r else 0, convert_biters / m if m else 0]
ax.bar(["roots", "converts"], shares, color="lightgray", edgecolor="black")
for x, (share, biters, total) in enumerate(zip(shares, [T, convert_biters], [r, m])):
    ax.text(x, share + 0.01, f"{biters} of {total}", ha="center")
ax.set_ylim(0, max(shares + [0.1]) * 1.3)
ax.set_ylabel("share who bit someone")
ax.set_title("Who bit someone, among finished vampires")
plt.show()

## Step 5: the test, and how its p-value is computed

### 5a. What the test asks

- **The list.** Write each vampire's count $y_i$ in one list: the $r$ roots first, then the converts: those bitten by roots first, then those bitten by those converts, and so on. Being a root means being in one of the first $r$ places.
- **The null hypothesis.** Every vampire's count comes from the same distribution, independently, roots and converts alike. We assume nothing about the shape of that distribution: equal biting, a few doing most, anything.
- **The alternative.** Converts are less likely than roots to bite anyone.
- **The statistic** is $T = Z_1 + \dots + Z_r$, the number of biters among the first $r$ places of the list, which are the roots (Step 3). The other $b - T$ biters are converts. A large $T$ is evidence that converts bite less.
- **The p-value is conditional on the counts.** We keep the counts we saw and rearrange only who has which. The reason: if a vampire bites $y$ people with chance $P(y)$, a list's chance is the product of $P$ over its counts, so every list made of the *same* counts has the same chance, whatever $P$ is, and we do not know $P$. Lists made of other counts have chances we cannot compute. This is how every permutation test works: the values stay, the labels move.
- **Possible lists only.** A convert exists only because someone bit them, so in the list, every convert must have been bitten by a vampire above them. A list where that fails cannot happen, and is left out. Among the possible lists made of our counts, each is equally likely under the null hypothesis.

Write $T_{\text{obs}}$ for the observed value of $T$: the value Step 3 computed from the export this notebook read (in practice mode, the made-up export). The p-value is the share of possible lists, made of the observed counts, whose $T$ is **equal to $T_{\text{obs}}$ or larger**:

$$p = \Pr(T \ge T_{\text{obs}}).$$

### 5b. A small example

Roots A and B, converts C and D, so there were 2 bites. Say A bit C and B bit D: the counts are 1, 1, 0, 0, and $T = 2$. Six lists can be made of these counts; three are possible:

$$
\begin{array}{cc|cc|c}
\text{A} & \text{B} & \text{1st convert} & \text{2nd convert} & T \\
\hline
1 & 1 & 0 & 0 & 2 \\
1 & 0 & 1 & 0 & 1 \\
0 & 1 & 1 & 0 & 1
\end{array}
$$

The other three, (1, 0, 0, 1), (0, 1, 0, 1), and (0, 0, 1, 1), put a bite on a convert who was never bitten. One of the three possible lists has $T \ge 2$, so $p = 1/3$.

### 5c. The formula, when no convert bit

When no convert bit, $T = b$: every biter is a root. The chance of that, over the possible lists, is

$$p = \frac{r-1}{V-1}\cdot\frac{r-2}{V-2}\cdots, \qquad \text{with } b - 1 \text{ factors.}$$

With one biter there are no factors, and $p = 1$: with one biter, the biter is always a root, which tells us nothing. In the example, $r = 2$, $V = 4$, $b = 2$, and $p = \frac{1}{3}$. The reason for the formula is a counting result called the *cycle lemma*: of all lists, a fraction $r/V$ are possible, and every list with all its biters among the roots is one of them. This is `compare.p_trees(r, b, V)`.

When a convert bit, there is no formula this simple, and Step 6 gives the p-value.

In [ ]:
# Step 5c: the exact p-value, when no convert bit
if convert_biters == 0 and b >= 1:
    factors = [f"{r - i}/{V - i}" for i in range(1, b)]
    p_exact = compare.p_trees(r, b, V)
    print(f"No convert bit: T = b = {b}, r = {r}, V = {V}")
    print("p = " + (" x ".join(factors) if factors else "1 (one biter)") + f" = {show_p(p_exact)}")
elif b == 0:
    p_exact = 1.0
    print("Nobody bit: T = 0, and every list has T = 0, so p = 1.")
else:
    p_exact = None
    print(f"{convert_biters} convert(s) bit, so the formula does not apply. Step 6 gives the p-value.")

## Step 6: the p-value from random possible lists

This step draws possible lists at random, each possible list made of our counts equally likely, and counts:

1. Shuffle all $V$ counts into a random order.
2. A list can be started at any of its $V$ places and read round to the place before it. Of those $V$ readings, exactly $r$ are possible lists (the cycle lemma again). Take one of those $r$ at random.
3. Its first $r$ places are the roots: count the biters among them. That is one random $T$.
4. Repeat `REPS` times. The share of random lists with $T$ equal to $T_{\text{obs}}$ or larger is the p-value.

This is `compare.p_trees_sampled`. When no convert bit, it checks the formula of Step 5c and should be close to it; if it is not, something is wrong. When a convert bit, it is the p-value.

In [ ]:
# Step 6: random possible lists
if m == 0:
    print("No converts are counted, so there is nothing to compare.")
else:
    p_sampled = compare.p_trees_sampled(root_counts, convert_counts, REPS, random.Random(6))
    print(f"Share of {REPS} random possible lists with T at least {T}: {show_p(p_sampled, REPS)}"
          + (f"   (Step 5c: {show_p(p_exact)})" if p_exact is not None else ""))

## Step 7: what the test does not say

- **That converts bite as often as roots.** No result can show that. With $p$ of 0.05 or more, a report says that with $m$ converts the test cannot tell converts who never bite from converts who bite as often as roots, in those words.
- **Why converts bite less, if $p$ is small.** Roots recruited by a walker turned at the moment they joined, with the walker beside them. Converts turned 12 hours after being bitten, usually alone. A difference between the groups may be a difference between those two situations rather than between the people. A report says so, whatever the result.
- **The day.** A convert always turned later in the game than the root who bit them. If everyone bit less as the game got older, converts would bite less without being any different, and this test cannot tell the two apart.
- **Which chains are counted.** A chain in which a convert bit has more generations, so it finishes later, and is more likely than other chains to be left out as not finished. Leaving such chains out makes converts look as if they bite less. *Reasoned November 4, 2026, and not checked by simulation.* It does no harm when the chains left out are unfinished only because their roots joined late.

## Step 8: your result

Fill this in, in your own words, in a text cell below:

- Whether poster roots are counted, and how many chains are left out and why (Step 3).
- $r$, $m$, $T$, the number of converts who bit, and $b$.
- The p-value: from the formula (Step 5c) if no convert bit, with the check (Step 6); otherwise from Step 6.
- What your plan said each result would mean, and which one happened.
- What the result does not say (Step 7).